# Web scraping

Aim of this notebook is to scrape information of start-ups and financial institution contained in this website: https://www.eu-startups.com

This information will be mainly used for visualization purposes.

In [ ]:
# Libraries

import requests
import pandas as pd
from bs4 import BeautifulSoup

from pathlib import Path
import os

Definining export path

In [8]:
repo_root = Path.cwd().parent
export_path = repo_root / "Data" / "00_Raw"

## EU Start-up database

In [ ]:
# Defining target URLS for Spain and Italy

URL_ITA = "https://www.eu-startups.com/directory/wpbdp_category/italian-startups/"
URL_ES = "https://www.eu-startups.com/directory/wpbdp_category/spanish-startups/"

URLS = [URL_ITA, URL_ES]

In [ ]:
# Extracting database entries to dict

all_startups_data = []

for base_url in URLS:
    current_url = base_url
    while current_url:
        try:
            page = requests.get(current_url)
            page.raise_for_status() # Raise an exception for HTTP errors
            soup = BeautifulSoup(page.content, "html.parser")
            results = soup.find(id="wpbdp-listings-list")

            if not results:
                print(f"No listings found on {current_url}. Skipping.")
                break # Break from inner pagination loop if no results container

            contents = results.find_all("div", class_="excerpt-content wpbdp-hide-title")

            if not contents:
                print(f"No startup cards found on {current_url}. Skipping.")
                break # Break from inner pagination loop if no content

            for content in contents:
                company_name = 'N/A'
                link_url = 'N/A'
                country = 'N/A'
                based_in = 'N/A'
                tags = 'N/A'
                founded = 'N/A'

                # Safely extract company name and link URL
                try:
                    anchors = content.find_all("a")
                    if len(anchors) > 1:
                        company_name = anchors[1].text.strip()
                    if len(anchors) > 0 and "href" in anchors[0].attrs:
                        link_url = anchors[0]["href"]
                except (IndexError, AttributeError, KeyError): # Handle if 'a' tags or 'href' attribute are missing
                    pass

                # Safely extract country
                try:
                    anchors = content.find_all("a")
                    if len(anchors) > 2:
                        country = anchors[2].text.strip()
                except (IndexError, AttributeError): # Handle if 'a' tags are missing
                    pass

                # Safely extract 'Based In'
                based_in_div = content.find("div", class_="wpbdp-field-based_in")
                if based_in_div:
                    value_div = based_in_div.find("div", class_="value")
                    if value_div:
                        based_in = value_div.text.strip()

                # Safely extract 'Tags'
                tags_div = content.find("div", class_="wpbdp-field-tags")
                if tags_div:
                    value_div = tags_div.find("div", class_="value")
                    if value_div:
                        tags = value_div.text.strip()

                # Safely extract 'Founded'
                founded_div = content.find("div", class_="wpbdp-field-founded")
                if founded_div:
                    value_div = founded_div.find("div", class_="value")
                    if value_div:
                        founded = value_div.text.strip()

                startup_info = {
                    "Company Name": company_name,
                    "URL": link_url,
                    "Country": country,
                    "Based In": based_in,
                    "Tags": tags,
                    "Founded": founded
                }
                all_startups_data.append(startup_info)

            # Handle pagination
            next_page_link = None
            pagination_div = results.find("div", class_="wpbdp-pagination")
            if pagination_div:
                next_span = pagination_div.find("span", class_="next")
                if next_span and next_span.find("a"):
                    next_page_link = next_span.find("a")["href"]
            current_url = next_page_link

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {current_url}: {e}. Skipping to next URL.")
            current_url = None # Break current pagination loop and move to next base_url
        except Exception as e:
            print(f"An unexpected error occurred while processing {current_url}: {e}. Skipping to next URL.")
            current_url = None # Break current pagination loop and move to next base_url

Error fetching https://www.eu-startups.com/directory/wpbdp_category/spanish-startups/page/11/: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Skipping to next URL.


In [ ]:
# Exporting to excel

df_full_startups = pd.DataFrame(all_startups_data)

export_path.mkdir(parents=True, exist_ok=True)
df_full_startups.to_excel(export_path / "EU_Startups_Full.xlsx", index=False)
print(f"Successfully collected {len(df_full_startups)} startups and saved to '{export_path / 'EU_Startups_Full.xlsx'}'")

Successfully collected 1645 startups and saved to 'c:\Users\albet\OneDrive\Documenti\Alberto\ZZ_Utility\Programs\VScode\Py_projects\TFM\Data\00_Raw\EU_Startups_Full.xlsx'


## EU Start_up investors

In [ ]:
# Defining urls

URL_ITA = "https://www.eu-startups.com/investor-location/?country=IT"
URL_ES = "https://www.eu-startups.com/investor-location/?country=ES"

URLS = [URL_ITA, URL_ES]

In [ ]:
# Extracting database entries to dict

all_investors_data = []

for base_url in URLS:
    current_url = base_url
    while current_url:
        try:
            page = requests.get(current_url)
            page.raise_for_status() # Raise an exception for HTTP errors
            soup = BeautifulSoup(page.content, "html.parser")

            # Find the main container for investor listings
            main_container = soup.find("div", class_="inv-single-details")

            if not main_container:
                print(f"No investor listings found on {current_url}. Skipping.")
                break # Break from inner pagination loop if no main container

            investor_titles = main_container.find_all("h1", class_="inv-single-title")

            if not investor_titles:
                print(f"No investor cards found on {current_url}. Skipping.")
                break # Break from inner pagination loop if no investor cards

            for title_element in investor_titles:
                investor_name = 'N/A'
                website_url = 'N/A'
                hq_location = 'N/A'
                investor_type = 'N/A'
                investment_areas = 'N/A'
                funding_stage = 'N/A'

                investor_name = title_element.text.strip()

                if title_element.parent and 'href' in title_element.parent.attrs:
                    website_url = title_element.parent['href']

                details_block = title_element.parent.find_next_sibling("div", class_="investor-details")

                if details_block:
                    location_div = details_block.find("div", class_="inv-single-location")
                    if location_div and location_div.find("span"):
                        hq_location = location_div.find("span").text.strip()

                    type_div = details_block.find("div", class_="inv-single-type")
                    if type_div and type_div.find("span"):
                        investor_type = type_div.find("span").text.strip()

                    areas_div = details_block.find("div", class_="inv-single-short")
                    if areas_div and areas_div.find("span"):
                        investment_areas = areas_div.find("span").text.strip()

                    investment_divs = details_block.find_all("div", class_="inv-single-investment")
                    for inv_div in investment_divs:
                        if inv_div.find("h5") and "Funding Stage:" in inv_div.find("h5").text:
                            if inv_div.find("span"):
                                funding_stage = inv_div.find("span").text.strip()
                                break

                investor_info = {
                    "Investor Name": investor_name,
                    "Website URL": website_url,
                    "HQ Location": hq_location,
                    "Investor Type": investor_type,
                    "Investment Areas": investment_areas,
                    "Funding Stage": funding_stage
                }
                all_investors_data.append(investor_info)

            # Handle pagination for investor pages
            next_page_link = None
            # The investor page pagination seems to be within a div with class 'pagination cpm-pagination'
            pagination_div = main_container.find("div", class_="pagination cpm-pagination")
            if pagination_div:
                # Look for a 'next page' link within the pagination div
                next_link_tag = pagination_div.find("a", class_="next page-numbers")
                if next_link_tag and "href" in next_link_tag.attrs:
                    next_page_link = next_link_tag["href"]
            current_url = next_page_link

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {current_url}: {e}. Skipping to next URL.")
            current_url = None # Break current pagination loop and move to next base_url
        except Exception as e:
            print(f"An unexpected error occurred while processing {current_url}: {e}. Skipping to next URL.")
            current_url = None # Break current pagination loop and move to next base_url

In [ ]:
# Exporting to excel

df_full_investors = pd.DataFrame(all_investors_data)
export_path.mkdir(parents=True, exist_ok=True)
df_full_investors.to_excel(export_path / "EU_Startups_Financial_Institution.xlsx", index=False)
print(f"Successfully collected {len(df_full_investors)} investors and saved to '{export_path / 'EU_Startups_Financial_Institution.xlsx'}'")

Successfully collected 36 investors and saved to 'c:\Users\albet\OneDrive\Documenti\Alberto\ZZ_Utility\Programs\VScode\Py_projects\TFM\Data\00_Raw\EU_Investors_Full.xlsx'
